# Notebook 02b — Re-ID: Ranking, Umbral (ROC/EER) y Demo de Acceso

**Jose Alfredo — Fase 2-B**

Este notebook demuestra:
1. Construcción y cache de la galería (usando `build_gallery` de Leandro).
2. Ranking por distancia coseno con `rank_identities`.
3. Análisis de umbral: distribuciones genuinos/impostores + curva ROC + EER.
4. Demo aceptar/rechazar para la defensa y el video.

> **Requisito previo:** Leandro debe haber corrido `embeddings.py` y tener la galería
> de fotos en `data/gallery/`. Si la galería no existe, se genera un conjunto
> sintético para demostrar el flujo (modo demo).

In [ ]:
# ── Setup: agregar raíz del proyecto al path ──────────────────────────────────
import sys, os
from pathlib import Path

# Subir hasta la carpeta raíz del proyecto (RutaCamba/)
root = Path(os.getcwd()).parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Raíz del proyecto: {root}")

In [ ]:
# ── Imports principales ───────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# Módulos del proyecto
from src import config
from src.reid.gallery import load_gallery, save_gallery, get_identity_names
from src.reid.ranking import rank_identities, cosine_distance
from src.reid.access import verify_identity, reload_gallery

print("✅ Imports correctos")
print(f"   REID_THRESHOLD de config: {config.REID_THRESHOLD}")
print(f"   GALLERY_DIR: {config.GALLERY_DIR}")

---
## 1. Cargar (o simular) la Galería

Intentamos cargar la galería real generada por Leandro (`build_gallery`).
Si no existe, generamos embeddings sintéticos para demostrar el flujo.

In [ ]:
GALLERY_CACHE = os.path.join(root, config.GALLERY_DIR, "gallery_cache.pkl")

gallery = load_gallery(cache_path=GALLERY_CACHE)

DEMO_MODE = False

if gallery is None:
    print("⚠️  Galería real no encontrada. Generando galería SINTÉTICA para demostración.")
    print("   (Cuando Leandro corra build_gallery(), reemplazá esto por la galería real)")
    DEMO_MODE = True

    # ── Galería sintética: 5 identidades, 4 fotos cada una, embeddings ArcFace 512-d
    np.random.seed(42)
    IDENTITIES = ["diego", "leandro", "jose_alfredo", "alejandro", "nicole"]
    EMBED_DIM = 512

    gallery = {}
    base_vectors = {}  # vector central por identidad

    for name in IDENTITIES:
        base = np.random.randn(EMBED_DIM).astype(np.float32)
        base = base / np.linalg.norm(base)  # normalizar
        base_vectors[name] = base

        # 4 fotos: vector base + ruido pequeño (simula variación de ángulo/iluminación)
        embeddings = []
        for _ in range(4):
            noise = np.random.randn(EMBED_DIM).astype(np.float32) * 0.12
            emb = base + noise
            emb = emb / np.linalg.norm(emb)
            embeddings.append(emb)
        gallery[name] = embeddings

    save_gallery(gallery, cache_path=GALLERY_CACHE)
    reload_gallery()
    print(f"\n✅ Galería sintética creada y cacheada: {len(gallery)} identidades")

else:
    print(f"✅ Galería REAL cargada: {len(gallery)} identidades")
    base_vectors = None  # no disponibles en galería real

print("\nIdentidades registradas:")
for name in get_identity_names(gallery):
    print(f"  · {name}: {len(gallery[name])} embedding(s)")

---
## 2. Demo de Ranking por Distancia Coseno

In [ ]:
# ── Caso 1: probe = la misma persona (genuino) ────────────────────────────────
if DEMO_MODE:
    # En modo demo usamos el vector base + pequeño ruido como selfie
    probe_genuine = base_vectors["jose_alfredo"] + np.random.randn(512).astype(np.float32) * 0.08
    probe_genuine /= np.linalg.norm(probe_genuine)
    declared_id_genuine = "jose_alfredo"
else:
    # TODO (con galería real): cargar una selfie de prueba y obtener embedding
    # from src.reid.embeddings import get_embedding
    # probe_genuine = get_embedding("ruta/a/selfie_test.jpg")
    # declared_id_genuine = "nombre_real"
    raise ValueError("En modo galería real: cargar probe desde imagen.")

ranked = rank_identities(probe_genuine, gallery, top_k=5)

print(f"🔍 Probe: selfie de '{declared_id_genuine}'")
print(f"{'Rank':<6} {'Identidad':<20} {'Distancia coseno':<18} {'¿Match?':<10}")
print("-" * 58)
for i, (ident, dist) in enumerate(ranked):
    match = "✅" if ident == declared_id_genuine else ""
    marker = "← TOP-1" if i == 0 else ""
    print(f"{i+1:<6} {ident:<20} {dist:<18.4f} {match:<10} {marker}")

print(f"\nUmbral actual (config): {config.REID_THRESHOLD}")
print(f"Distancia top-1:        {ranked[0][1]:.4f}")
print(f"Acceso concedido:       {'✅ SÍ' if ranked[0][0] == declared_id_genuine and ranked[0][1] < config.REID_THRESHOLD else '❌ NO'}")

---
## 3. Análisis de Umbral: Distribuciones y Curva ROC/EER

Para justificar el umbral generamos todos los pares posibles de la galería:
- **Genuinos (intra-clase):** par de fotos de la MISMA persona.
- **Impostores (inter-clase):** par de fotos de personas DISTINTAS.

In [ ]:
from itertools import combinations

genuine_dists = []   # distancias par mismo-mismo
impostor_dists = []  # distancias par mismo-distinto

identity_names = get_identity_names(gallery)

# Pares genuinos: todas las combinaciones de embeddings dentro de la misma identidad
for name in identity_names:
    embs = gallery[name]
    for i, j in combinations(range(len(embs)), 2):
        d = cosine_distance(embs[i], embs[j])
        genuine_dists.append(d)

# Pares impostores: un embedding de cada identidad vs un embedding de otra
for n1, n2 in combinations(identity_names, 2):
    for emb1 in gallery[n1]:
        for emb2 in gallery[n2]:
            d = cosine_distance(emb1, emb2)
            impostor_dists.append(d)

genuine_dists = np.array(genuine_dists)
impostor_dists = np.array(impostor_dists)

print(f"Pares genuinos:   {len(genuine_dists)}  | media={genuine_dists.mean():.4f} std={genuine_dists.std():.4f}")
print(f"Pares impostores: {len(impostor_dists)} | media={impostor_dists.mean():.4f} std={impostor_dists.std():.4f}")

In [ ]:
# ── Distribuciones genuinos vs impostores ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Fase 2-B — Distribuciones de distancia coseno (Re-ID)", fontsize=14, fontweight='bold')

# Histogramas superpuestos
ax = axes[0]
bins = np.linspace(0, 1.5, 60)
ax.hist(genuine_dists, bins=bins, alpha=0.65, color='#2ecc71', label=f'Genuinos (n={len(genuine_dists)})')
ax.hist(impostor_dists, bins=bins, alpha=0.65, color='#e74c3c', label=f'Impostores (n={len(impostor_dists)})')
ax.axvline(config.REID_THRESHOLD, color='#f39c12', linewidth=2.5, linestyle='--',
           label=f'Umbral actual = {config.REID_THRESHOLD}')
ax.set_xlabel("Distancia coseno", fontsize=11)
ax.set_ylabel("Frecuencia", fontsize=11)
ax.set_title("Distribución de distancias genuinos vs impostores", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Curva ROC
# label=1 → genuino, label=0 → impostor
# score = 1 - distancia (mayor = más similar)
labels = np.concatenate([np.ones(len(genuine_dists)), np.zeros(len(impostor_dists))])
scores = np.concatenate([1 - genuine_dists, 1 - impostor_dists])

fpr, tpr, thresholds_roc = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

# EER: punto donde FPR ≈ FNR = 1 - TPR
fnr = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - fnr))
eer_value = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold_sim = thresholds_roc[eer_idx]  # threshold en espacio similitud
eer_threshold_dist = 1 - eer_threshold_sim   # convertir a distancia coseno

ax2 = axes[1]
ax2.plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'ROC (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax2.scatter(fpr[eer_idx], tpr[eer_idx], color='#e74c3c', zorder=5, s=100,
            label=f'EER = {eer_value:.3f} @ dist≈{eer_threshold_dist:.3f}')
ax2.set_xlabel("False Positive Rate", fontsize=11)
ax2.set_ylabel("True Positive Rate", fontsize=11)
ax2.set_title("Curva ROC — Verificación de identidad", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(root, "docs", "reid_roc_threshold.png"), dpi=120, bbox_inches='tight')
plt.show()

print(f"\n📊 Resultados del análisis:")
print(f"   AUC-ROC:           {roc_auc:.4f}")
print(f"   EER:               {eer_value:.4f}  ({eer_value*100:.2f}%)")
print(f"   Umbral EER (dist): {eer_threshold_dist:.4f}")
print(f"   Umbral actual:     {config.REID_THRESHOLD}")
print(f"\n💡 {'✅ El umbral actual es consistente con EER.' if abs(config.REID_THRESHOLD - eer_threshold_dist) < 0.1 else '⚠️  Considera actualizar REID_THRESHOLD en config.py a ' + str(round(eer_threshold_dist, 3))}")

---
## 4. Demo de Acceso: Caso ACEPTA y Caso RECHAZA

> Esta sección es la que se muestra en el video y la defensa.

In [ ]:
def print_access_result(result: dict, declared_id: str, case_label: str):
    """Pretty-print del resultado de verify_identity."""
    icon = "✅ ACCESO CONCEDIDO" if result["access"] else "❌ ACCESO DENEGADO"
    print(f"\n{'='*55}")
    print(f"  {case_label}")
    print(f"{'='*55}")
    print(f"  Identidad declarada : {declared_id}")
    print(f"  Top-1 identificado  : {result['top1_identity']}")
    print(f"  Distancia coseno    : {result['distance']:.4f}")
    print(f"  Umbral              : {config.REID_THRESHOLD}")
    print(f"  Resultado           : {icon}")
    print(f"\n  Top-5 ranking:")
    for i, (ident, dist) in enumerate(result["topk"]):
        marker = " ← top-1" if i == 0 else ""
        print(f"    {i+1}. {ident:<20} dist={dist:.4f}{marker}")
    if "error" in result:
        print(f"  ⚠️  Error: {result['error']}")
    print()


if DEMO_MODE:
    # ── CASO 1: ACEPTA — probe de la misma persona ────────────────────────────
    probe_accept = base_vectors["nicole"] + np.random.randn(512).astype(np.float32) * 0.08
    probe_accept /= np.linalg.norm(probe_accept)

    # Inyectamos el probe sintético directamente al ranking (sin pasar por DeepFace)
    ranked_accept = rank_identities(probe_accept, gallery, top_k=5)
    result_accept = {
        "access": ranked_accept[0][0] == "nicole" and ranked_accept[0][1] < config.REID_THRESHOLD,
        "distance": ranked_accept[0][1],
        "top1_identity": ranked_accept[0][0],
        "topk": ranked_accept,
    }
    print_access_result(result_accept, "nicole", "🟢 CASO 1 — MISMA PERSONA (debería ACEPTAR)")

    # ── CASO 2: RECHAZA — probe de una persona diferente (impostor) ───────────
    probe_reject = base_vectors["diego"] + np.random.randn(512).astype(np.float32) * 0.08
    probe_reject /= np.linalg.norm(probe_reject)

    ranked_reject = rank_identities(probe_reject, gallery, top_k=5)
    # El impostor declara ser 'nicole' pero su top-1 es 'diego'
    result_reject = {
        "access": ranked_reject[0][0] == "nicole" and ranked_reject[0][1] < config.REID_THRESHOLD,
        "distance": next(d for n, d in ranked_reject if n == "nicole"),
        "top1_identity": ranked_reject[0][0],
        "topk": ranked_reject,
    }
    print_access_result(result_reject, "nicole", "🔴 CASO 2 — IMPOSTOR (debería RECHAZAR)")

else:
    print("Con galería real: usar verify_identity() directamente con imágenes reales.")
    print("Ejemplo:")
    print("  result = verify_identity('nombre_declarado', 'ruta/selfie.jpg')")
    print("  print_access_result(result, 'nombre_declarado', 'Demo real')")

In [ ]:
# ── Visualización de los dos casos ────────────────────────────────────────────
if DEMO_MODE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("Demo de verificación de identidad — Acepta vs Rechaza", fontsize=13, fontweight='bold')

    for ax, (result, declared, title, color) in zip(
        axes,
        [
            (result_accept, "nicole",  "✅ Caso ACEPTA (misma persona)",  '#2ecc71'),
            (result_reject, "nicole",  "❌ Caso RECHAZA (impostor)",       '#e74c3c'),
        ]
    ):
        names = [t[0] for t in result['topk']]
        dists = [t[1] for t in result['topk']]
        bar_colors = ['#f39c12' if n == declared else '#95a5a6' for n in names]

        bars = ax.barh(names, dists, color=bar_colors, edgecolor='white', linewidth=0.8)
        ax.axvline(config.REID_THRESHOLD, color='#c0392b', linewidth=2, linestyle='--',
                   label=f'Umbral = {config.REID_THRESHOLD}')
        ax.set_xlabel("Distancia coseno")
        ax.set_title(title, fontsize=11, color=color, fontweight='bold')
        ax.legend(fontsize=9)
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)

        # Anotar la identidad declarada
        for bar, name in zip(bars, names):
            if name == declared:
                ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                        f" declarado", va='center', fontsize=8, color='#f39c12')

    plt.tight_layout()
    plt.savefig(os.path.join(root, "docs", "reid_demo_acceso.png"), dpi=120, bbox_inches='tight')
    plt.show()

print("\n📸 Gráficas guardadas en docs/ para el informe.")

---
## 5. Resumen de Métricas y Umbral Recomendado

> Copiar estos valores en la bitácora `docs/decisiones/fase2_decisiones_jose.md` (Decisión 003).

In [ ]:
print("=" * 55)
print("  RESUMEN FASE 2-B — Jose Alfredo")
print("=" * 55)
print(f"  Galería: {'SINTÉTICA (demo)' if DEMO_MODE else 'REAL'}")
print(f"  Identidades registradas: {len(gallery)}")
print(f"  Embeddings totales: {sum(len(v) for v in gallery.values())}")
print()
print(f"  Pares genuinos:   {len(genuine_dists)}  → μ={genuine_dists.mean():.4f}  σ={genuine_dists.std():.4f}")
print(f"  Pares impostores: {len(impostor_dists)} → μ={impostor_dists.mean():.4f}  σ={impostor_dists.std():.4f}")
print()
print(f"  AUC-ROC:           {roc_auc:.4f}")
print(f"  EER:               {eer_value:.4f} ({eer_value*100:.2f}%)")
print(f"  Umbral EER (dist): {eer_threshold_dist:.4f}")
print(f"  Umbral en config:  {config.REID_THRESHOLD}")
print()
if abs(config.REID_THRESHOLD - eer_threshold_dist) > 0.05:
    print(f"  ⚠️  RECOMENDACIÓN: actualizar REID_THRESHOLD a {eer_threshold_dist:.3f} en src/config.py")
else:
    print(f"  ✅ REID_THRESHOLD = {config.REID_THRESHOLD} es consistente con EER ({eer_threshold_dist:.3f})")
print("=" * 55)

---
## 6. Notas para la Integración con Nicole (Fase 6)

Para conectar el endpoint `/verify` de la API:

```python
# api/main.py (Nicole)
from src.reid.access import verify_identity

@app.post("/verify")
async def verify(declared_id: str, selfie: UploadFile):
    image = PIL.Image.open(selfie.file)
    result = verify_identity(declared_id, image)
    return result
    # result = {"access": bool, "distance": float, "top1_identity": str, "topk": [...]}
```

**Requisito previo:** la galería debe estar construida y cacheada en `data/gallery/gallery_cache.pkl`.
Esto lo hace Leandro con `build_gallery()` + mi `save_gallery()`.

Si Nicole necesita recargar la galería sin reiniciar el servidor:
```python
from src.reid.access import reload_gallery
reload_gallery()  # resetea el singleton, se recarga en el próximo request
```